### 🏗️ Advanced Architecture: OOP & AI Integration

In this phase 5.1, we move beyond simple scripts to a **Professional Software Architecture**:

* **Inheritance Hierarchy:** We defined a `Taxi` base class. `ElectricTaxi` and `GasTaxi` inherit common properties (ID, Driver) but implement their own unique driving behavior.
* **Composition:** Instead of making the AI model part of the inheritance, we "composed" it. A Taxi **has a** model engine. This makes the system modular and flexible.
* **UUID & Random:** Used to simulate a real-world fleet environment where every vehicle is unique and assigned to a driver automatically.
* **Logic Separation:** The `ElectricTaxi` handles energy concerns, while the `GasTaxi` handles fuel, but both utilize the same **AI Intelligence** for trip quality prediction.

In [1]:
import pandas as pd
import uuid
import random
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from load_cleaned_dataset import get_cleaned_df
df = get_cleaned_df()


# Ignore warnings for clean output
warnings.filterwarnings("ignore")

# ==========================================
# 1. DATA PREPARATION & MODEL TRAINING
# ==========================================
# Assuming 'df' is your loaded NYC Taxi dataset
# Let's prepare the features and target for training
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['is_generous'] = (df['tip_amount'] > (df['fare_amount'] * 0.20)).astype(int)

# Features: Fare, Distance, Hour, Passengers, Payment Type
X = df[['fare_amount', 'trip_distance', 'pickup_hour', 'passenger_count', 'payment_type']]
y = df['is_generous']

# Train the Classifier (The Brain of our System)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf_model = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
clf_model.fit(X_train, y_train)

# ==========================================
# 2. OOP STRUCTURE (Based on your Images)
# ==========================================

# Composition Class: The AI Model component
class TippingModel:
    def __init__(self, trained_model):
        self.model = trained_model

    def predict_quality(self, data_dict):
        # Convert dictionary to DataFrame to match training format
        df_input = pd.DataFrame([data_dict])
        prediction = self.model.predict(df_input)[0]
        return "High Quality (Generous)" if prediction == 1 else "Normal Quality"

# Base Class (Inheritance Hierarchy)
class Taxi:
    def __init__(self, model_component):
        self.id = uuid.uuid4() # Unique ID using UUID
        self.driver = random.choice(["Hans", "Anna", "Lukas", "Sarah", "Elena"]) # Random Driver
        self.ai_engine = model_component # Composition: Taxi "has a" Model

    def get_info(self):
        return f"Taxi ID: {self.id} | Driver: {self.driver}"

# Specialized Subclasses (Inheritance)
class ElectricTaxi(Taxi):
    def drive(self):
        return "Driving silently on battery power... 🔋"

class GasTaxi(Taxi):
    def drive(self):
        return "Driving on gasoline engine... ⛽"

# ==========================================
# 3. LIVE EXECUTION (The Example)
# ==========================================

# Initialize the AI Component with our trained model
my_ai_component = TippingModel(clf_model)

# Create an instance of an Electric Taxi
my_ev_taxi = ElectricTaxi(my_ai_component)

print("\n" + "="*50)
print("🚀 NYC SMART FLEET INTELLIGENCE SYSTEM")
print("="*50)
print(my_ev_taxi.get_info())
print(f"Vehicle Type: {my_ev_taxi.drive()}")
print("-" * 50)

# Simulate trip data input
trip_data = {
    'fare_amount': 23.27,
    'trip_distance': 5.0,
    'pickup_hour': 14,
    'passenger_count': 1,
    'payment_type': 1
}

# The Taxi uses its internal AI engine to predict quality
quality = my_ev_taxi.ai_engine.predict_quality(trip_data)

print(f"Trip Input: {trip_data['trip_distance']} miles | Fare: ${trip_data['fare_amount']}")
print(f"AI Prediction: This is a {quality} trip.")
print("="*50)

 Loading Dataset: ../data/raw\nyc-yellow-taxi-trip-records-january-2024\nyc_tlc_yellow_2024_01_cleaned.csv
 Dataset Loaded Successfully! Total Rows: 2,599,399
Type enforcement complete. New Memory Usage:
399.12 MB

🚀 NYC SMART FLEET INTELLIGENCE SYSTEM
Taxi ID: c2e9f930-dbbb-4542-aafc-36f62f8ad135 | Driver: Elena
Vehicle Type: Driving silently on battery power... 🔋
--------------------------------------------------
Trip Input: 5.0 miles | Fare: $23.27
AI Prediction: This is a High Quality (Generous) trip.


# 📍 phase 5.2. Finding High-Demand Routes (Heatmap Analysis)

### 🕵️‍♂️ Key High-Demand Patterns
To find the most requested routes, we analyze the `PULocationID` (Pick-up) and `DOLocationID` (Drop-off). Based on NYC data, the "Golden Triangle" of demand is:

1.  **Airport Shuttles:** Routes between **JFK/LaGuardia Airports** and **Manhattan**. These are long-distance, high-fare, and constant.
2.  **The Commuter Flow:** Morning peak hours (7 AM - 9 AM) from residential areas (Upper East/West Side) to the **Financial District**.
3.  **The Nightlife Pulse:** Late-night demand (10 PM - 2 AM) centered around **Times Square, Chelsea, and the Meatpacking District**.

### 📊 Data Strategy
To implement this in your code, you should use **Group-By** operations to count occurrences of Pick-up/Drop-off pairs.

In [2]:
# Function to find the top 5 most frequent routes
def get_top_routes(df):
    # Grouping by Pick-up and Drop-off locations
    route_counts = df.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
    
    # Sorting to find the highest demand
    top_routes = route_counts.sort_values(by='trip_count', ascending=False).head(5)
    
    return top_routes

# --- Execution ---
top_5 = get_top_routes(df)

print("\n" + "🔥 TOP 5 HIGH-DEMAND ROUTES")
print("="*35)
for index, row in top_5.iterrows():
    print(f"Route: {int(row['PULocationID'])} ➡️ {int(row['DOLocationID'])} | Trips: {row['trip_count']}")


🔥 TOP 5 HIGH-DEMAND ROUTES
Route: 237 ➡️ 236 | Trips: 20929
Route: 236 ➡️ 237 | Trips: 18354
Route: 236 ➡️ 236 | Trips: 14460
Route: 237 ➡️ 237 | Trips: 13362
Route: 161 ➡️ 237 | Trips: 9697


### 📊 Strategic Route Analysis

* **Dominant Cluster:** The data shows a massive concentration in zones **236 (Upper East Side North)** and **237 (Upper East Side South)**. 
* **High Frequency:** With over **20,000 trips** on a single route, these are "Gold Mine" sectors for taxi drivers.
* **Operational Strategy:** For our `ElectricTaxi` fleet, these routes are ideal because the distances are short, preserving battery life while maximizing the number of fares (Turnover).
* **Commuter Logic:** The connection between **161 (Midtown)** and **237** indicates a strong flow of professionals moving between the business district and residential luxury zones.

# 🛡️ Phase 5.3. Secure Fleet Ledger: Implementing Blockchain Technology

To ensure the integrity of trip records and payments, we integrate a **Decentralized Ledger System** (Blockchain) into our Taxi project.

### 🧩 Core Components
* **Cryptographic Hashing:** Every trip record is transformed into a unique digital fingerprint (Hash) using the `hashlib` library.
* **Block Class:** Represents a single trip or a batch of trips, containing a timestamp, the trip data (driver, fare, route), the previous block's hash, and its own hash.
* **Blockchain Class:** Manages the chain of blocks. It ensures that any attempt to alter a previous trip will break the entire chain, making the system tamper-proof.

### 💡 Why Blockchain for Taxis?
1. **Transparency:** Passengers and drivers can verify the exact fare and distance without disputes.
2. **Security:** Once a trip is recorded in a block, it cannot be deleted or changed.
3. **Auditability:** Perfect for regulatory compliance and financial auditing of taxi revenues.

In [3]:
import hashlib
import json
from time import time

class Block:
    """Represents a single block in the ledger containing trip data"""
    def __init__(self, index, trip_data, previous_hash):
        self.index = index
        self.timestamp = time()
        self.trip_data = trip_data  # Example: {'driver': 'Lukas', 'fare': 23.27}
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        """Creates a SHA-256 hash of the block contents"""
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "trip_data": self.trip_data,
            "previous_hash": self.previous_hash
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

class TaxiBlockchain:
    """Manages the chain of trip records"""
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        """Initial block of the blockchain"""
        return Block(0, "Genesis Block - System Start", "0")

    def get_latest_block(self):
        return self.chain[-1]

    def add_trip_record(self, trip_data):
        """Adds a new trip to the ledger after calculating its hash"""
        new_block = Block(
            index=len(self.chain),
            trip_data=trip_data,
            previous_hash=self.get_latest_block().hash
        )
        self.chain.append(new_block)

    def is_chain_valid(self):
        """Verifies if the ledger has been tampered with"""
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i-1]

            # Check if current hash is still correct
            if current.hash != current.calculate_hash():
                return False
            # Check if current block correctly points to previous hash
            if current.previous_hash != previous.hash:
                return False
        return True

# --- EXECUTION ---
taxi_ledger = TaxiBlockchain()

# Recording Trips
print("📡 Recording new trips in the Blockchain...")
taxi_ledger.add_trip_record({"driver": "Anna", "fare": 25.50, "route": "237 -> 236"})
taxi_ledger.add_trip_record({"driver": "Lukas", "fare": 12.00, "route": "161 -> 237"})

# Displaying Ledger
for block in taxi_ledger.chain:
    print(f"\nBlock #{block.index}")
    print(f"Hash: {block.hash}")
    print(f"Data: {block.trip_data}")

# Security Check
print(f"\nIs the trip ledger secure? {taxi_ledger.is_chain_valid()}")

# Attempting to tamper (simulating a hack)
taxi_ledger.chain[1].trip_data = {"driver": "Anna", "fare": 1000.00} # Faking a fare
print(f"ALERT: Tampering detected! Is the ledger still valid? {taxi_ledger.is_chain_valid()}")

📡 Recording new trips in the Blockchain...

Block #0
Hash: edfdf3270241260737303c0c30b2f14ac294066d63b46f3e5ccddc0f9ff837ea
Data: Genesis Block - System Start

Block #1
Hash: 55ea15b5878e33359ff5786cb5f806abdd2c9bddcdfab1e3f3134c6854d649cb
Data: {'driver': 'Anna', 'fare': 25.5, 'route': '237 -> 236'}

Block #2
Hash: c9e56ee840b1171c3f72988b197d7a88d0c01efefaea101aa2faf43659ce3dc2
Data: {'driver': 'Lukas', 'fare': 12.0, 'route': '161 -> 237'}

Is the trip ledger secure? True
ALERT: Tampering detected! Is the ledger still valid? False


# 💎 Phase 5.4. Smart Rewards: Integrating AI Quality with Blockchain Ledger

In this final phase, we create a **Incentivization Ecosystem**. The system automatically rewards drivers based on the AI's "High Quality" prediction.

### 🛠️ Strategic Integration
* **AI-Led Rewards:** If the `TippingModel` predicts a "High Quality" trip, the system automatically triggers a reward function.
* **Immutable Proof:** The reward is recorded as a transaction within a Block. Since the Blockchain is tamper-proof, the driver’s rewards are secure and verifiable.
* **Smart Fleet Governance:** This encourages drivers to maintain high service standards to earn more digital tokens.

### 🏗️ Architectural Flow
1. **Trip Input** -> 2. **AI Analysis** -> 3. **Quality Result** -> 4. **Blockchain Transaction** -> 5. **Token Awarded**.

In [4]:
# --- BLOCKCHAIN & REWARD SYSTEM ---

class Block:
    def __init__(self, index, data, previous_hash):
        self.index = index
        self.timestamp = time()
        self.data = data # Contains trip info and tokens awarded
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps(self.data, sort_keys=True) + str(self.timestamp) + self.previous_hash
        return hashlib.sha256(block_string.encode()).hexdigest()

class RewardBlockchain:
    def __init__(self):
        self.chain = [self.create_genesis()]
        self.token_balance = {} # Tracks driver rewards

    def create_genesis(self):
        return Block(0, {"info": "Genesis Block"}, "0")

    def add_reward_block(self, driver, quality, fare):
        # Logic: High Quality = 10 Tokens, Normal = 2 Tokens
        tokens = 10 if quality == "High Quality (Generous)" else 2
        
        # Update Balance
        self.token_balance[driver] = self.token_balance.get(driver, 0) + tokens
        
        trip_entry = {
            "driver": driver,
            "quality": quality,
            "fare": fare,
            "tokens_awarded": tokens
        }
        
        new_block = Block(len(self.chain), trip_entry, self.chain[-1].hash)
        self.chain.append(new_block)
        return tokens

# --- SIMULATING THE FULL CYCLE ---

# 1. Setup Blockchain
fleet_ledger = RewardBlockchain()

# 2. Assume we have our AI result from the previous step
current_driver = "Anna"
predicted_quality = "High Quality (Generous)" # From AI Model
actual_fare = 23.27

# 3. Process Reward into Blockchain
print(f"🎬 Processing Trip for {current_driver}...")
awarded = fleet_ledger.add_reward_block(current_driver, predicted_quality, actual_fare)

print("-" * 40)
print(f"✅ Blockchain Updated!")
print(f"⭐ Quality: {predicted_quality}")
print(f"💰 Tokens Awarded: {awarded}")
print(f"🏦 {current_driver}'s Total Balance: {fleet_ledger.token_balance[current_driver]} Tokens")
print("-" * 40)

# 4. Verify Ledger Integrity
print(f"🔐 Is Ledger Valid & Secure? {all(fleet_ledger.chain[i].hash == fleet_ledger.chain[i].calculate_hash() for i in range(1, len(fleet_ledger.chain)))}")

🎬 Processing Trip for Anna...
----------------------------------------
✅ Blockchain Updated!
⭐ Quality: High Quality (Generous)
💰 Tokens Awarded: 10
🏦 Anna's Total Balance: 10 Tokens
----------------------------------------
🔐 Is Ledger Valid & Secure? True


# 🏆 Phase 5.4.1 Final Integration: AI Analysis & Secure Token Rewards

Based on the latest system execution, the driver **Anna** was assigned a high-value trip. The system has been updated to integrate the **Reward Logic** directly into the **Blockchain Ledger**.

### 🛠️ Key Logic Updates:
* **Dynamic Driver Handling:** The system now tracks individual balances for drivers like Anna.
* **Automated Rewarding:** Upon predicting a "High Quality" trip, the Blockchain records a transaction awarding **10 Tokens** to the driver's secure wallet.
* **Integrity Check:** The ledger confirms that the 10 tokens awarded to Anna are cryptographically linked to her specific Taxi ID and Trip Fare.

In [5]:
# --- BLOCKCHAIN & REWARD CLASSES ---

class Block:
    def __init__(self, index, data, previous_hash):
        self.index = index
        self.timestamp = time()
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps(self.data, sort_keys=True) + str(self.timestamp) + self.previous_hash
        return hashlib.sha256(block_string.encode()).hexdigest()

class TaxiBlockchain:
    def __init__(self):
        self.chain = [self.create_genesis()]
        self.balances = {} # Wallet balances for drivers

    def create_genesis(self):
        return Block(0, {"info": "Genesis Block"}, "0")

    def add_reward(self, driver, taxi_id, quality):
        # High Quality = 10 Tokens, Normal = 2 Tokens
        token_amount = 10 if "High Quality" in quality else 2
        
        # Update driver balance
        self.balances[driver] = self.balances.get(driver, 0) + token_amount
        
        transaction = {
            "taxi_id": str(taxi_id),
            "driver": driver,
            "quality": quality,
            "tokens_awarded": token_amount
        }
        
        new_block = Block(len(self.chain), transaction, self.chain[-1].hash)
        self.chain.append(new_block)
        return token_amount

# --- THE SMART FLEET SYSTEM ---

# 1. Initialize Blockchain Ledger
ledger = TaxiBlockchain()

# 2. Results from your execution
taxi_id = "55aaefce-292c-4995-8cd9-eed48b2b614e"
driver_name = "Anna"
vehicle_type = "ElectricTaxi 🔋"
predicted_quality = "High Quality (Generous)"
fare_amount = 23.27

print("\n" + "="*50)
print("🚀 NYC SMART FLEET - REWARD SYSTEM ACTIVATED")
print("="*50)
print(f"Taxi ID: {taxi_id} | Driver: {driver_name}")
print(f"Status: {vehicle_type}")
print(f"AI Prediction: {predicted_quality}")
print("-" * 50)

# 3. Secure the reward in the Blockchain
tokens = ledger.add_reward(driver_name, taxi_id, predicted_quality)

print(f"🔗 BLOCKCHAIN TRANSACTION COMPLETE")
print(f"✅ Awarded: {tokens} Tokens to {driver_name}")
print(f"🏦 New Wallet Balance: {ledger.balances[driver_name]} Tokens")
print(f"🛡️ Current Block Hash: {ledger.chain[-1].hash[:20]}...")
print("="*50)

# 4. Final Verification
if ledger.chain[-1].calculate_hash() == ledger.chain[-1].hash:
    print("🔒 Data Integrity Verified: Reward record is Immutable.")


🚀 NYC SMART FLEET - REWARD SYSTEM ACTIVATED
Taxi ID: 55aaefce-292c-4995-8cd9-eed48b2b614e | Driver: Anna
Status: ElectricTaxi 🔋
AI Prediction: High Quality (Generous)
--------------------------------------------------
🔗 BLOCKCHAIN TRANSACTION COMPLETE
✅ Awarded: 10 Tokens to Anna
🏦 New Wallet Balance: 10 Tokens
🛡️ Current Block Hash: 0302d6cc0a673425e9b1...
🔒 Data Integrity Verified: Reward record is Immutable.
